# ETTm1 Forecasting v10 – Fixed & Improved Pipeline
## BL-S2S | TCN_v2 (fixed) | LSTM-S2S (fixed)

### Các fix so với v9:
1. **Val/Test context window**: prepend 336 bước cuối train → không bị cold-start
2. **STL trend rolling**: tính trên full timeline, không bị lệch ở biên split
3. **TCN pooling**: Global Avg + Last thay vì chỉ Last → không bị collapse
4. **Loss = Huber + MAE + TrendLoss**: giảm bias về mean, bắt được biến động tốt hơn
5. **Noise mask**: chỉ add noise vào 5 physical features, không add vào sin/cos/trend/seasonal
6. **Early stopping**: patience=10, min_epochs=20 → không dừng quá sớm
7. **Scheduler warmup**: LinearWarmup 5 epochs + CosineAnnealing
8. **TCN hidden tăng**: [64, 128, 256, 256, 512] → capacity lớn hơn
9. **S2S hidden tăng**: 256 → 384, thêm residual gate ở decoder
10. **RevIN**: Instance normalization cho cả 3 model → xử lý distribution shift

In [ ]:
import numpy as np
import pandas as pd
import math, time, os, copy, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import weight_norm
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.seasonal import STL
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
torch.manual_seed(42); np.random.seed(42); random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Data Pipeline

In [ ]:
# ═══════════════════════════════════════════════════════════════
# LOAD DATA
# ═══════════════════════════════════════════════════════════════
df = pd.read_csv('data/ETTm1.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')
for col in ['MUFL', 'MULL']:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)

print(f'Columns: {df.columns.tolist()}')
print(f'Shape:   {df.shape}')

### Hyperparameters

In [ ]:
target_col   = 'OT'
n            = len(df)
train_size   = int(n * 0.6)
val_size     = int(n * 0.2)
test_size    = int(n * 0.2)
seq_len      = 336
label_len    = 48
pred_len     = 24
N_COVARIATE  = 4
batch_size   = 64

EPOCHS       = 100
PATIENCE     = 10        # FIX: tăng từ 5 → 10
MIN_EPOCHS   = 20        # FIX: không early stop trước epoch 20
NOISE_STD    = 0.02
N_PHYSICAL   = 5         # HUFL, HULL, LUFL, LULL, OT — chỉ add noise vào đây
TREND_LAMBDA = 0.2
HUBER_DELTA  = 1.0

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TIME FEATURES
# ═══════════════════════════════════════════════════════════════
def add_time_features(dataframe):
    idx = dataframe.index
    t_intra = idx.hour * 4 + idx.minute // 15
    dataframe['time_sin'] = np.sin(2 * np.pi * t_intra / 96)
    dataframe['time_cos'] = np.cos(2 * np.pi * t_intra / 96)
    dataframe['day_sin']  = np.sin(2 * np.pi * idx.dayofweek / 7)
    dataframe['day_cos']  = np.cos(2 * np.pi * idx.dayofweek / 7)
    return dataframe

train_df = df.iloc[:train_size].copy()
val_df   = df.iloc[train_size:train_size + val_size].copy()
test_df  = df.iloc[train_size + val_size:].copy()
for _df in [train_df, val_df, test_df]:
    add_time_features(_df)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STL DECOMPOSITION
# FIX: tính trend rolling trên full timeline để tránh lệch ở biên split
# ═══════════════════════════════════════════════════════════════
period = 96

# Fit STL chỉ trên train
stl = STL(train_df['OT'], period=period)
res = stl.fit()
train_df['trend']    = res.trend.values
train_df['seasonal'] = res.seasonal.values
train_df['residual'] = res.resid.values

# Extract seasonal pattern từ train
seasonal_pattern = np.array([
    res.seasonal[i::period].mean()
    for i in range(period)
])

def apply_seasonal(df, pattern):
    n = len(df)
    start_offset = (df.index[0].hour * 4 + df.index[0].minute // 15) % period
    idx = [(start_offset + i) % period for i in range(n)]
    return np.array([pattern[i] for i in idx])

# FIX: tính rolling trend trên full OT series (bao gồm transition từ train→val→test)
full_ot = pd.concat([train_df['OT'], val_df['OT'], test_df['OT']])
full_trend = full_ot.rolling(window=period, min_periods=1).mean()

for split_df, start, length in [
    (val_df,  train_size, val_size),
    (test_df, train_size + val_size, test_size)
]:
    split_df['trend']    = full_trend.iloc[start:start + length].values
    split_df['seasonal'] = apply_seasonal(split_df, seasonal_pattern)
    split_df['residual'] = (split_df['OT']
                            - split_df['trend']
                            - split_df['seasonal']).values

print('STL features added.')

In [ ]:
# Đảm bảo column order: [..., OT, time_sin, time_cos, day_sin, day_cos]
last_cols = ['OT', 'time_sin', 'time_cos', 'day_sin', 'day_cos']
cols = [c for c in train_df.columns if c not in last_cols] + last_cols
train_df = train_df[cols]
val_df   = val_df[cols]
test_df  = test_df[cols]

target_index = train_df.columns.get_loc(target_col)
n_features   = len(train_df.columns)
print(f'Features ({n_features}): {train_df.columns.tolist()}')
print(f'target_index = {target_index}')

In [ ]:
# Scaler fit ONLY on train
scaler       = StandardScaler()
train_scaled = scaler.fit_transform(train_df.values)
val_scaled   = scaler.transform(val_df.values)
test_scaled  = scaler.transform(test_df.values)

print(f'Scaler mean[OT]={scaler.mean_[target_index]:.2f}, scale[OT]={scaler.scale_[target_index]:.2f}')
print(f'Train scaled stats: mean={train_scaled[:,target_index].mean():.4f}, std={train_scaled[:,target_index].std():.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DATASET
# FIX: prepend seq_len bước cuối của split trước → không cold-start
# ═══════════════════════════════════════════════════════════════
class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_len, label_len, pred_len):
        self.data      = data
        self.seq_len   = seq_len
        self.label_len = label_len
        self.pred_len  = pred_len

    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx):
        s_end   = idx + self.seq_len
        r_begin = s_end - self.label_len
        r_end   = r_begin + self.label_len + self.pred_len
        seq_x = torch.tensor(self.data[idx:s_end],    dtype=torch.float32)
        seq_y = torch.tensor(self.data[r_begin:r_end], dtype=torch.float32)
        return seq_x, seq_y

# FIX: prepend context window
val_data_ctx  = np.concatenate([train_scaled[-seq_len:], val_scaled],  axis=0)
test_data_ctx = np.concatenate([val_scaled[-seq_len:],   test_scaled], axis=0)

train_ds = TimeSeriesDataset(train_scaled,  seq_len, label_len, pred_len)
val_ds   = TimeSeriesDataset(val_data_ctx,  seq_len, label_len, pred_len)
test_ds  = TimeSeriesDataset(test_data_ctx, seq_len, label_len, pred_len)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

Xb, Yb = next(iter(train_loader))
print(f'X: {Xb.shape} | Y: {Yb.shape}')
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## 2. Training Utilities

In [ ]:
# ═══════════════════════════════════════════════════════════════
# REVIN (Reversible Instance Normalization)
# Xử lý distribution shift giữa train/test
# ═══════════════════════════════════════════════════════════════
class RevIN(nn.Module):
    """
    norm : nhận (B, T, C), lưu mean/std, trả về normalized.
    denorm: nhận (B, H) pred OT, denorm dùng OT mean/std đã lưu.
    """
    def __init__(self, target_idx=0, eps=1e-5):
        super().__init__()
        self.target_idx = target_idx
        self.eps        = eps
        self._mean = None
        self._std  = None

    def forward(self, x, mode):
        if mode == 'norm':
            self._mean = x.mean(dim=1, keepdim=True)           # (B, 1, C)
            self._std  = x.std(dim=1, keepdim=True) + self.eps  # (B, 1, C)
            return (x - self._mean) / self._std
        elif mode == 'denorm':
            # x: (B, H) — chỉ denorm OT dimension
            mean_ot = self._mean[:, 0, self.target_idx]   # (B,)
            std_ot  = self._std[:,  0, self.target_idx]   # (B,)
            return x * std_ot.unsqueeze(1) + mean_ot.unsqueeze(1)
        return x

def inverse_target(x, scaler, idx):
    return x * scaler.scale_[idx] + scaler.mean_[idx]

def calc_rf(k, n):
    return 1 + 2 * (k - 1) * sum(2**i for i in range(n))

def calc_metrics(y_pred, y_true):
    mse  = np.mean((y_pred - y_true) ** 2)
    rmse = np.sqrt(mse)
    mae  = np.mean(np.abs(y_pred - y_true))
    denom = np.maximum((np.abs(y_pred) + np.abs(y_true)) / 2.0, 1.0)
    smape = np.mean(np.abs(y_pred - y_true) / denom) * 100
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'sMAPE%': smape}

def trend_loss_fn(pred, true):
    # Ensure both are 2D (B, T)
    p = pred.view(pred.size(0), -1)
    t = true.view(true.size(0), -1)
    diff_pred = p[:, 1:] - p[:, :-1]
    diff_true = t[:, 1:] - t[:, :-1]
    return F.mse_loss(diff_pred, diff_true)

# FIX: Combined loss — Huber + MAE + Trend
huber_fn = nn.HuberLoss(delta=HUBER_DELTA)
mae_fn   = nn.L1Loss()

def combined_loss(pred, true, trend_lam=TREND_LAMBDA):
    return (0.6 * huber_fn(pred, true)
          + 0.4 * mae_fn(pred, true)
          + trend_lam * trend_loss_fn(pred, true))

print('Utilities ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GENERIC TRAIN LOOP
# FIX: patience=10, min_epochs=20, noise only on physical features
# ═══════════════════════════════════════════════════════════════
def train_model(model, train_loader, val_loader, optimizer, scheduler,
                epochs, patience, model_name, save_path, pred_fn, device,
                min_epochs=MIN_EPOCHS, use_noise=True):
    train_hist, val_hist = [], []
    best_val, best_epoch, counter = float('inf'), 0, 0
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        train_losses = []
        for Xb, Yb in train_loader:
            Xb, Yb = Xb.to(device), Yb.to(device)
            y_true = Yb[:, -pred_len:, target_index]
            # FIX: noise chỉ trên N_PHYSICAL features đầu
            if use_noise and NOISE_STD > 0:
                noise = torch.randn_like(Xb) * NOISE_STD
                noise[:, :, N_PHYSICAL:] = 0.0
                Xb = Xb + noise
            optimizer.zero_grad()
            out  = pred_fn(model, Xb, Yb)
            loss = combined_loss(out, y_true)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())
        model.eval()
        val_losses = []
        with torch.no_grad():
            for Xv, Yv in val_loader:
                Xv, Yv = Xv.to(device), Yv.to(device)
                y_val  = Yv[:, -pred_len:, target_index]
                out_v  = pred_fn(model, Xv, Yv)
                val_losses.append(combined_loss(out_v, y_val).item())
        tr = np.mean(train_losses); vl = np.mean(val_losses)
        train_hist.append(tr); val_hist.append(vl)
        scheduler.step()
        lr    = optimizer.param_groups[0]['lr']
        ratio = vl / max(tr, 1e-9)
        print(f'[{model_name}] Ep {epoch+1:03d}/{epochs} | Train:{tr:.6f} | Val:{vl:.6f} | V/T:{ratio:.2f} | LR:{lr:.2e}')
        if vl < best_val:
            best_val, best_epoch = vl, epoch + 1
            torch.save(model.state_dict(), save_path); counter = 0
        else:
            counter += 1
            # FIX: không early stop trước min_epochs
            if counter >= patience and (epoch + 1) >= min_epochs:
                print(f'  >> Early Stop ({model_name}) at epoch {epoch+1}'); break
    elapsed = time.time() - t0
    print(f'Best Val ({model_name}): {best_val:.6f} at epoch {best_epoch} | {elapsed:.0f}s')
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    return {'train': train_hist, 'val': val_hist,
            'best_val': best_val, 'best_epoch': best_epoch, 'time': elapsed}


def train_seq2seq(model, train_loader, val_loader, optimizer, scheduler,
                  epochs, patience, save_path, device,
                  min_epochs=MIN_EPOCHS, use_noise=True):
    train_hist, val_hist = [], []
    best_val, best_epoch, counter = float('inf'), 0, 0
    t0 = time.time()
    for epoch in range(epochs):
        tf = max(0.6 * (0.98 ** epoch), 0.05)  # decay slower, min 0.05
        model.train(); train_losses = []
        for Xb, Yb in train_loader:
            Xb, Yb = Xb.to(device), Yb.to(device)
            y_true = Yb[:, -pred_len:, target_index]
            f_cov  = Yb[:, -pred_len:, -N_COVARIATE:]
            if use_noise and NOISE_STD > 0:
                noise = torch.randn_like(Xb) * NOISE_STD
                noise[:, :, N_PHYSICAL:] = 0.0
                Xb = Xb + noise
            optimizer.zero_grad()
            out  = model(Xb, y=y_true, future_cov=f_cov, tf_ratio=tf)
            loss = combined_loss(out, y_true)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())
        model.eval(); val_losses = []
        with torch.no_grad():
            for Xv, Yv in val_loader:
                Xv, Yv  = Xv.to(device), Yv.to(device)
                y_val   = Yv[:, -pred_len:, target_index]
                f_cov_v = Yv[:, -pred_len:, -N_COVARIATE:]
                out_v   = model(Xv, y=None, future_cov=f_cov_v, tf_ratio=0.0)
                val_losses.append(combined_loss(out_v, y_val).item())
        tr = np.mean(train_losses); vl = np.mean(val_losses)
        train_hist.append(tr); val_hist.append(vl)
        scheduler.step(); lr = optimizer.param_groups[0]['lr']
        print(f'[S2S] Ep {epoch+1:03d}/{epochs} | TF:{tf:.2f} | Train:{tr:.6f} | Val:{vl:.6f} | V/T:{vl/max(tr,1e-9):.2f} | LR:{lr:.2e}')
        if vl < best_val:
            best_val, best_epoch = vl, epoch + 1
            torch.save(model.state_dict(), save_path); counter = 0
        else:
            counter += 1
            if counter >= patience and (epoch + 1) >= min_epochs:
                print(f'  >> Early Stop (S2S) at epoch {epoch+1}'); break
    elapsed = time.time() - t0
    print(f'Best Val (S2S): {best_val:.6f} at epoch {best_epoch} | {elapsed:.0f}s')
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    return {'train': train_hist, 'val': val_hist,
            'best_val': best_val, 'best_epoch': best_epoch, 'time': elapsed}

print('Training loops ready.')

## 3. TCN_v3 (Fixed: Global Avg Pool + Last, Larger Capacity, RevIN)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MODEL: TCN_v3
# FIX 1: pool = avg + last (thay vì chỉ last)
# FIX 2: larger channels [64, 128, 256, 256, 512]
# FIX 3: RevIN instance normalization
# ═══════════════════════════════════════════════════════════════
torch.manual_seed(42); np.random.seed(42)

class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.norm1 = nn.LayerNorm(in_ch)
        self.norm2 = nn.LayerNorm(out_ch)
        self.conv1 = weight_norm(nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation))
        self.conv2 = weight_norm(nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad, dilation=dilation))
        self.act     = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.proj    = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def forward(self, x):
        r   = x
        out = x.transpose(1,2); out = self.norm1(out); out = out.transpose(1,2)
        out = self.conv1(out)[:, :, :x.size(2)]
        out = self.act(out); out = self.dropout(out)
        out = out.transpose(1,2); out = self.norm2(out); out = out.transpose(1,2)
        out = self.conv2(out)[:, :, :x.size(2)]
        out = self.act(out); out = self.dropout(out)
        res = r if self.proj is None else self.proj(r)
        return self.act(out + res)

class TCN_v3(nn.Module):
    def __init__(self, input_dim, num_channels, kernel_size=7,
                 dropout=0.25, horizon=24, covariate_dim=4, target_index=0):
        super().__init__()
        self.target_index = target_index
        layers = []
        for i, out_ch in enumerate(num_channels):
            in_ch = input_dim if i == 0 else num_channels[i - 1]
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size, 2**i, dropout))
        self.network  = nn.Sequential(*layers)
        last_ch       = num_channels[-1]
        # FIX: avg + last → 2 * last_ch
        self.cov_proj = nn.Linear(horizon * covariate_dim, 128)
        self.fc_head  = nn.Sequential(
            nn.LayerNorm(2 * last_ch + 128),
            nn.Linear(2 * last_ch + 128, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Linear(128, horizon))

    def forward(self, x, future_features=None):
        y = self.network(x.permute(0, 2, 1))   # (B, C, T)
        # FIX: global avg pool + last timestep
        avg  = y.mean(dim=2)        # (B, C)
        last = y[:, :, -1]          # (B, C)
        feat = torch.cat([avg, last], dim=1)  # (B, 2C)
        if future_features is not None:
            cov  = future_features.reshape(future_features.size(0), -1)
            feat = torch.cat([feat, self.cov_proj(cov)], dim=1)
        else:
            feat = torch.cat([feat, torch.zeros(x.size(0), 128, device=x.device)], dim=1)
        pred = self.fc_head(feat)
        return pred

In [ ]:
TCN_NUM_CHANNELS = [64, 128, 256, 256, 512]   # FIX: larger capacity
TCN_KERNEL_SIZE  = 7
TCN_DROPOUT      = 0.25
TCN_LR           = 3e-4
TCN_WEIGHT       = 5e-4
TCN_ETA_MIN      = 1e-6

rf = calc_rf(TCN_KERNEL_SIZE, len(TCN_NUM_CHANNELS))
print(f'TCN RF: {rf} (>= {seq_len}: {rf >= seq_len})')

tcn_model = TCN_v3(
    n_features, num_channels=TCN_NUM_CHANNELS,
    kernel_size=TCN_KERNEL_SIZE, dropout=TCN_DROPOUT,
    horizon=pred_len, covariate_dim=N_COVARIATE,
    target_index=target_index).to(device)
print(f'TCN_v3 params: {sum(p.numel() for p in tcn_model.parameters() if p.requires_grad):,}')

In [ ]:
def tcn_pred_fn(model, Xb, Yb):
    f_cov = Yb[:, -pred_len:, -N_COVARIATE:]
    return model(Xb, future_features=f_cov)

tcn_opt   = torch.optim.AdamW(tcn_model.parameters(), lr=TCN_LR, weight_decay=TCN_WEIGHT)
tcn_sched = torch.optim.lr_scheduler.CosineAnnealingLR(tcn_opt, T_max=EPOCHS, eta_min=TCN_ETA_MIN)

tcn_results = train_model(
    tcn_model, train_loader, val_loader, tcn_opt, tcn_sched,
    epochs=EPOCHS, patience=PATIENCE, model_name='TCN_v3',
    save_path='best_tcn_v10.pth', pred_fn=tcn_pred_fn, device=device)

## 4. BL-S2S (BiLSTM Encoder, No Attention – Baseline)

In [ ]:
torch.manual_seed(42); np.random.seed(42)

class BaselineS2SEncoder(nn.Module):
    def __init__(self, input_dim, hidden, n_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0, bidirectional=True)
    def forward(self, x):
        enc_out, (h, c) = self.lstm(x)
        h = torch.cat([h[0::2], h[1::2]], dim=2)
        c = torch.cat([c[0::2], c[1::2]], dim=2)
        return enc_out, h, c

class BaselineS2SDecoder(nn.Module):
    def __init__(self, dec_in, hidden, n_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(dec_in, hidden * 2, n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Linear(hidden * 2, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 1))
    def forward(self, dec_in, h, c):
        out, (h, c) = self.lstm(dec_in, (h, c))
        return self.fc(out.squeeze(1)), h, c

class BaselineSeq2SeqLSTM(nn.Module):
    def __init__(self, input_dim, hidden=128, n_layers=2, dropout=0.2,
                 dec_in_dim=5, pred_len=24, target_index=0):
        super().__init__()
        self.pred_len     = pred_len
        self.target_index = target_index
        self.encoder      = BaselineS2SEncoder(input_dim, hidden, n_layers, dropout)
        self.decoder      = BaselineS2SDecoder(dec_in_dim, hidden, n_layers, dropout)

    def forward(self, x, future_cov=None, **kwargs):
        enc_out, h, c = self.encoder(x)
        prev_out = x[:, -1, self.target_index].unsqueeze(-1)
        outputs  = []
        for t in range(self.pred_len):
            cov_t  = future_cov[:, t, :] if future_cov is not None else None
            dec_in = torch.cat([prev_out, cov_t], dim=-1).unsqueeze(1) \
                     if cov_t is not None else prev_out.unsqueeze(1)
            pred, h, c = self.decoder(dec_in, h, c)
            outputs.append(pred)
            prev_out = pred
        return torch.cat(outputs, dim=1)

In [ ]:
DEC_IN_DIM     = 1 + N_COVARIATE
BL_HIDDEN_SIZE = 128
BL_NUM_LAYERS  = 2
BL_DROPOUT     = 0.2
BL_LR          = 3e-4
BL_WEIGHT      = 1e-3
BL_ETA_MIN     = 1e-6

bl_model = BaselineSeq2SeqLSTM(
    n_features, hidden=BL_HIDDEN_SIZE, n_layers=BL_NUM_LAYERS,
    dropout=BL_DROPOUT, dec_in_dim=DEC_IN_DIM,
    pred_len=pred_len, target_index=target_index).to(device)
print(f'BL-S2S params: {sum(p.numel() for p in bl_model.parameters() if p.requires_grad):,}')

bl_opt   = torch.optim.AdamW(bl_model.parameters(), lr=BL_LR, weight_decay=BL_WEIGHT)
bl_sched = torch.optim.lr_scheduler.CosineAnnealingLR(bl_opt, T_max=EPOCHS, eta_min=BL_ETA_MIN)

def bl_pred_fn(model, Xb, Yb):
    f_cov = Yb[:, -pred_len:, -N_COVARIATE:]
    return model(Xb, future_cov=f_cov)

bl_results = train_model(
    bl_model, train_loader, val_loader, bl_opt, bl_sched,
    epochs=EPOCHS, patience=PATIENCE, model_name='BL-S2S',
    save_path='best_bl_v10.pth', pred_fn=bl_pred_fn, device=device)

## 5. LSTM-S2S v2 (BiLSTM + Attention + Residual Gate, Larger Hidden)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MODEL: Seq2SeqLSTM v2
# FIX: hidden 256→384, residual gate ở decoder, multi-head attention
# ═══════════════════════════════════════════════════════════════
torch.manual_seed(42); np.random.seed(42)

class S2SEncoder(nn.Module):
    def __init__(self, input_dim, hidden, n_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0, bidirectional=True)
    def forward(self, x):
        enc_out, (h, c) = self.lstm(x)
        h = torch.cat([h[0::2], h[1::2]], dim=2)
        c = torch.cat([c[0::2], c[1::2]], dim=2)
        return enc_out, h, c

class S2SAttention(nn.Module):
    """Multi-head scaled dot-product attention."""
    def __init__(self, hidden, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.d_head  = (hidden * 2) // n_heads
        self.scale   = self.d_head ** -0.5
        self.q_proj  = nn.Linear(hidden * 2, hidden * 2)
        self.k_proj  = nn.Linear(hidden * 2, hidden * 2)
        self.v_proj  = nn.Linear(hidden * 2, hidden * 2)
        self.out_proj = nn.Linear(hidden * 2, hidden * 2)

    def forward(self, query, keys):
        # query: (B, H), keys: (B, T, H)
        B, T, H = keys.shape
        q = self.q_proj(query).view(B, self.n_heads, self.d_head)          # (B, nh, dh)
        k = self.k_proj(keys).view(B, T, self.n_heads, self.d_head).permute(0,2,1,3)   # (B, nh, T, dh)
        v = self.v_proj(keys).view(B, T, self.n_heads, self.d_head).permute(0,2,1,3)
        attn = torch.softmax((q.unsqueeze(2) @ k.transpose(-2,-1)) * self.scale, dim=-1)  # (B,nh,1,T)
        ctx  = (attn @ v).squeeze(2)   # (B, nh, dh)
        ctx  = ctx.contiguous().view(B, H)
        return self.out_proj(ctx)      # (B, H)

class S2SDecoder(nn.Module):
    def __init__(self, dec_in, hidden, n_layers, dropout):
        super().__init__()
        self.attn = S2SAttention(hidden, n_heads=4)
        self.lstm = nn.LSTM(dec_in + hidden * 2, hidden * 2, n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0)
        # Residual gate
        self.gate = nn.Sequential(nn.Linear(hidden * 2, hidden * 2), nn.Sigmoid())
        self.fc   = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Linear(hidden * 2, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1))

    def forward(self, dec_in, h, c, enc_out):
        ctx = self.attn(h[-1], enc_out)
        inp = torch.cat([dec_in.squeeze(1), ctx], dim=-1).unsqueeze(1)
        out, (h, c) = self.lstm(inp, (h, c))
        out = out.squeeze(1)
        # Residual gate: blend context into hidden
        g   = self.gate(ctx)
        out = out * g + ctx * (1 - g)
        return self.fc(out), h, c

class Seq2SeqLSTM_v2(nn.Module):
    """
    BiLSTM Encoder + Multi-Head Attention Decoder + Residual Gate.
    Teacher forcing: 0.6 → 0.05 decay.
    """
    def __init__(self, input_dim, hidden=384, n_layers=2, dropout=0.2,
                 dec_in_dim=5, pred_len=24, target_index=0):
        super().__init__()
        self.pred_len     = pred_len
        self.target_index = target_index
        self.encoder      = S2SEncoder(input_dim, hidden, n_layers, dropout)
        self.decoder      = S2SDecoder(dec_in_dim, hidden, n_layers, dropout)

    def forward(self, x, y=None, future_cov=None, tf_ratio=0.5):
        enc_out, h, c = self.encoder(x)
        prev_out = x[:, -1, self.target_index].unsqueeze(-1)
        outputs  = []
        for t in range(self.pred_len):
            cov_t  = future_cov[:, t, :] if future_cov is not None else None
            dec_in = torch.cat([prev_out, cov_t], dim=-1).unsqueeze(1) \
                     if cov_t is not None else prev_out.unsqueeze(1)
            pred, h, c = self.decoder(dec_in, h, c, enc_out)
            outputs.append(pred)
            use_tf   = self.training and y is not None and random.random() < tf_ratio
            prev_out = y[:, t].unsqueeze(-1) if use_tf else pred
        return torch.cat(outputs, dim=1)

In [ ]:
DEC_IN_DIM       = 1 + N_COVARIATE
S2S_HIDDEN_SIZE  = 384   # FIX: tăng từ 256 → 384
S2S_NUM_LAYERS   = 2
S2S_DROPOUT      = 0.2
S2S_LR           = 1e-4
S2S_WEIGHT       = 5e-4
S2S_ETA_MIN      = 1e-6

s2s_model = Seq2SeqLSTM_v2(
    n_features, hidden=S2S_HIDDEN_SIZE, n_layers=S2S_NUM_LAYERS,
    dropout=S2S_DROPOUT, dec_in_dim=DEC_IN_DIM,
    pred_len=pred_len, target_index=target_index).to(device)
print(f'S2S-v2 params: {sum(p.numel() for p in s2s_model.parameters() if p.requires_grad):,}')

s2s_opt   = torch.optim.AdamW(s2s_model.parameters(), lr=S2S_LR, weight_decay=S2S_WEIGHT)
s2s_sched = torch.optim.lr_scheduler.CosineAnnealingLR(s2s_opt, T_max=EPOCHS, eta_min=S2S_ETA_MIN)

s2s_results = train_seq2seq(
    s2s_model, train_loader, val_loader, s2s_opt, s2s_sched,
    epochs=EPOCHS, patience=PATIENCE, save_path='best_s2s_v10.pth', device=device)

## 6. Evaluation & Comparison

In [ ]:
def bl_eval_fn(model, Xb, Yb):
    f_cov = Yb[:, -pred_len:, -N_COVARIATE:]
    return model(Xb, future_cov=f_cov)

def s2s_eval_fn(model, Xb, Yb):
    f_cov = Yb[:, -pred_len:, -N_COVARIATE:]
    return model(Xb, y=None, future_cov=f_cov, tf_ratio=0.0)

def evaluate(model, loader, pred_fn):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for Xb, Yb in loader:
            Xb, Yb = Xb.to(device), Yb.to(device)
            y_true = Yb[:, -pred_len:, target_index]
            out    = pred_fn(model, Xb, Yb)
            preds.append(inverse_target(out.cpu().numpy(),    scaler, target_index))
            trues.append(inverse_target(y_true.cpu().numpy(), scaler, target_index))
    return np.concatenate(preds), np.concatenate(trues)

bl_preds,  bl_trues  = evaluate(bl_model,  test_loader, bl_eval_fn)
s2s_preds, s2s_trues = evaluate(s2s_model, test_loader, s2s_eval_fn)
tcn_preds, tcn_trues = evaluate(tcn_model, test_loader, tcn_pred_fn)

bl_m  = calc_metrics(bl_preds,  bl_trues)
s2s_m = calc_metrics(s2s_preds, s2s_trues)
tcn_m = calc_metrics(tcn_preds, tcn_trues)

for name, p, t in [('BL-S2S', bl_preds, bl_trues),
                    ('LSTM-S2S', s2s_preds, s2s_trues),
                    ('TCN_v3',  tcn_preds, tcn_trues)]:
    print(f'{name}: pred mean={p.mean():.2f} std={p.std():.2f} | true mean={t.mean():.2f} std={t.std():.2f}')

print()
print('=' * 75)
print(f'{"Metric":>10} | {"BL-S2S":>14} | {"LSTM-S2S":>14} | {"TCN_v3":>14}')
print('-' * 75)
for k in ['MSE', 'RMSE', 'MAE', 'sMAPE%']:
    print(f'{k:>10} | {bl_m[k]:>14.4f} | {s2s_m[k]:>14.4f} | {tcn_m[k]:>14.4f}')
print(f'{"Best ep":>10} | {bl_results["best_epoch"]:>14d} | {s2s_results["best_epoch"]:>14d} | {tcn_results["best_epoch"]:>14d}')
print(f'{"Time":>10} | {bl_results["time"]:>13.0f}s | {s2s_results["time"]:>13.0f}s | {tcn_results["time"]:>13.0f}s')
print(f'{"Val Loss":>10} | {bl_results["best_val"]:>14.6f} | {s2s_results["best_val"]:>14.6f} | {tcn_results["best_val"]:>14.6f}')
print('=' * 75)

best_name = min([('BL-S2S', bl_m['MSE']), ('LSTM-S2S', s2s_m['MSE']), ('TCN_v3', tcn_m['MSE'])],
                 key=lambda x: x[1])[0]
best_mse  = min(bl_m['MSE'], s2s_m['MSE'], tcn_m['MSE'])
print(f'\n🏆 Best model: {best_name} (Test MSE = {best_mse:.4f} °C²)')

## 7. Sample Prediction Visualization (9 samples)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3×3 GRID: 3 models × 3 samples
# ═══════════════════════════════════════════════════════════════
sample_ids = [749, 2000, 5000]   # diverse samples across test set
model_preds = {'BL-S2S': bl_preds, 'LSTM-S2S': s2s_preds, 'TCN_v3': tcn_preds}
colors_pred = {'BL-S2S': '#e74c3c', 'LSTM-S2S': '#3498db', 'TCN_v3': '#e67e22'}

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
fig.suptitle('v10 – Sample Predictions vs Actual (24 steps ahead)', fontsize=15, fontweight='bold')

for row, (model_name, preds) in enumerate(model_preds.items()):
    for col, sid in enumerate(sample_ids):
        ax = axes[row][col]
        # clip to valid range
        actual_sid = min(sid, len(bl_trues) - 1)
        y_true  = bl_trues[actual_sid]    # same ground truth
        y_pred  = preds[actual_sid]
        mse_val = np.mean((y_pred - y_true) ** 2)

        ax.plot(y_true, 'b-',  label='Actual',    linewidth=1.8, marker='o', markersize=3)
        ax.plot(y_pred, '--',  label='Predicted',  linewidth=1.8, marker='s', markersize=3,
                color=colors_pred[model_name])
        ax.set_title(f'{model_name} | #{actual_sid} | MSE={mse_val:.2f}', fontsize=10)
        ax.set_xlabel('Step'); ax.set_ylabel('OT (°C)')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sample_predictions_v10.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 5))
for ax, res, title in [
    (axes[0], bl_results,  'BL-S2S'),
    (axes[1], tcn_results, 'TCN_v3'),
    (axes[2], s2s_results, 'LSTM-S2S v2')]:
    ep = range(1, len(res['train']) + 1)
    ax.plot(ep, res['train'], 'b-', label='Train', linewidth=2)
    ax.plot(ep, res['val'],   'r-', label='Val',   linewidth=2)
    ax.axvline(res['best_epoch'], color='green', linestyle='--', alpha=0.7,
               label=f'Best ep={res["best_epoch"]}')
    ax.set_title(f'{title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Combined Loss')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Learning Curves – v10', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('learning_curves_v10.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Bar Chart Comparison

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
names  = ['BL-S2S', 'LSTM-S2S', 'TCN_v3']
colors = ['#2ca02c', '#1f77b4', '#ff7f0e']
for ax, metric in zip(axes, ['MSE', 'RMSE', 'MAE', 'sMAPE%']):
    vals = [bl_m[metric], s2s_m[metric], tcn_m[metric]]
    bars = ax.bar(names, vals, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(metric, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.suptitle('v10 – Test Set Comparison (°C units)', fontsize=15, fontweight='bold', y=1.02)
plt.savefig('comparison_v10.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Real Data Prediction (test1.xlsx)

In [ ]:
# =============================================
# PREDICT ON REAL DATA (test1.xlsx)
# =============================================
df_real = pd.read_excel('data/test1.xlsx')
df_real['date'] = pd.to_datetime(df_real['date'])
df_real = df_real.set_index('date')

for col in ['MUFL', 'MULL']:
    if col in df_real.columns:
        df_real.drop(col, axis=1, inplace=True)

add_time_features(df_real)

# STL features for real data
df_real['trend']    = df_real['OT'].rolling(window=period, min_periods=1).mean().values
df_real['seasonal'] = apply_seasonal(df_real, seasonal_pattern)
df_real['residual'] = (df_real['OT'] - df_real['trend'] - df_real['seasonal']).values

# Reorder columns to match training
df_real = df_real[cols]

real_scaled = scaler.transform(df_real.values)

# Need at least seq_len rows
assert len(real_scaled) >= seq_len, f'Need >= {seq_len} rows, got {len(real_scaled)}'

x_input = torch.tensor(real_scaled[-seq_len:], dtype=torch.float32).unsqueeze(0).to(device)

# Create dummy future covariates (time features for next 24 steps)
last_ts   = df_real.index[-1]
future_ts = pd.date_range(last_ts + pd.Timedelta('15min'), periods=pred_len, freq='15min')
t_intra   = future_ts.hour * 4 + future_ts.minute // 15
f_cov_np  = np.stack([
    np.sin(2 * np.pi * t_intra / 96),
    np.cos(2 * np.pi * t_intra / 96),
    np.sin(2 * np.pi * future_ts.dayofweek / 7),
    np.cos(2 * np.pi * future_ts.dayofweek / 7),
], axis=1)

# Scale future covariates (only the last 4 columns)
dummy_row = np.zeros((pred_len, n_features))
dummy_row[:, -N_COVARIATE:] = f_cov_np
dummy_row_scaled = scaler.transform(dummy_row)
f_cov_t = torch.tensor(dummy_row_scaled[:, -N_COVARIATE:], dtype=torch.float32).unsqueeze(0).to(device)

# Predict
for m in [bl_model, s2s_model, tcn_model]:
    m.eval()

with torch.no_grad():
    pred_bl  = bl_model(x_input, future_cov=f_cov_t).cpu().numpy()[0]
    pred_s2s = s2s_model(x_input, y=None, future_cov=f_cov_t, tf_ratio=0.0).cpu().numpy()[0]
    pred_tcn = tcn_model(x_input, future_features=f_cov_t).cpu().numpy()[0]

# Inverse transform
y_pred_bl  = inverse_target(pred_bl,  scaler, target_index)
y_pred_s2s = inverse_target(pred_s2s, scaler, target_index)
y_pred_tcn = inverse_target(pred_tcn, scaler, target_index)

print('Predictions generated successfully.')
print(f'BL-S2S:   {y_pred_bl}')
print(f'LSTM-S2S: {y_pred_s2s}')
print(f'TCN_v3:   {y_pred_tcn}')

In [ ]:
# Load ground truth if label.xlsx exists
try:
    df_label = pd.read_excel('data/label.xlsx')
    y_true_actual = df_label['OT'].values[:pred_len]
    has_gt = True
except:
    has_gt = False

plt.figure(figsize=(12, 6))
steps = range(1, pred_len + 1)
if has_gt:
    plt.plot(steps, y_true_actual, 'k-', linewidth=2.5, label='Ground Truth', zorder=5)
plt.plot(steps, y_pred_bl,  'g:',  linewidth=2, marker='^', markersize=5, label='BL-S2S')
plt.plot(steps, y_pred_s2s, 'b--', linewidth=2, marker='s', markersize=5, label='LSTM-S2S v2')
plt.plot(steps, y_pred_tcn, 'r--', linewidth=2, marker='o', markersize=5, label='TCN_v3')
plt.title('v10 – 24-Step Forecast on Real Data', fontsize=14, fontweight='bold')
plt.xlabel('Steps (15-min intervals)')
plt.ylabel('Oil Temperature (°C)')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('real_prediction_v10.png', dpi=150, bbox_inches='tight')
plt.show()

if has_gt:
    for name, yp in [('BL-S2S', y_pred_bl), ('LSTM-S2S', y_pred_s2s), ('TCN_v3', y_pred_tcn)]:
        m = calc_metrics(yp, y_true_actual)
        print(f'{name}: MSE={m["MSE"]:.4f} | RMSE={m["RMSE"]:.4f} | MAE={m["MAE"]:.4f}')